# SMOTE + Noise v3 — Correlation Structure Preserved

## Các vấn đề được phát hiện và sửa trong v3:

| # | Vấn đề | Root Cause | Fix |
|---|--------|-----------|-----|
| 1 | **NDVI_CV drift lớn nhất** (MAE=0.136) | Dùng `Std = Range/4` sai — thực tế `Std = Range × 0.454` | Tái tính đúng công thức |
| 2 | **Wind_Max bị thổi phồng** (corr drift 0.43) | Flag=1 trong gốc có Wind_Max=10.625 cố định, nhưng code gán random(11,35) | Khi flag=1: gán đúng 10.625 |
| 3 | **Correlation structure tổng thể** (Frobenius=2.51) | SMOTE không bảo toàn rank correlation | **Iman-Conover rank matching** sau SMOTE |
| 4 | **NDVI_Season_Std** kế thừa sai | Bị coi là derived từ Range, nhưng Range/4 ≠ thực tế | Tính lại từ `Range × 0.454` |
| 5 | **Moisture_Ratio / Rootzone_Diff drift** | sm_surface/sm_rootzone variance bị thu hẹp | Iman-Conover tự sửa |


In [2]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import rankdata

# ============================================================
# 0. SEED & CẤU HÌNH
# ============================================================
SEED        = 42
NUM_ROWS    = 20000
K_NEIGHBORS = 5
NOISE_LEVEL = 0.015
np.random.seed(SEED)

# ============================================================
# 1. LOAD DỮ LIỆU
# ============================================================
print('[1/7] Đọc dữ liệu gốc...')
data = pd.read_csv('Agri_Data_Cleaned.csv')
print(f'     Gốc: {data.shape[0]:,} dòng × {data.shape[1]} cột')

CATEGORICAL_COLS = [
    'District', 'Season', 'Crop Name', 'Transplant',
    'Growth', 'Harvest', 'pH_Suitability',
    'Dominant_Soil_Texture', 'Water_Availability_Cat',
    'Extreme_Heat_Risk', 'Is_Extreme_Heat',
    'is_extreme_Heat_Stress_Days', 'is_extreme_Wind_Max'
]

# Các cột sẽ được TÁI TÍNH sau — không đưa vào SMOTE
DERIVED_COLS = [
    'CN_Ratio', 'Rain_Temp_Ratio',
    'Rootzone_Surface_Diff', 'Moisture_Ratio',
    'NDVI_Season_Range', 'NDVI_Season_CV', 'NDVI_Season_Std'
]

BASE_NUMERIC_COLS = [
    c for c in data.columns
    if c not in CATEGORICAL_COLS + DERIVED_COLS and c != 'Yield'
]

# Cột skewness cao → dùng log-space
LOG_COLS = ['Area', 'Production']

print(f'     Base numeric: {len(BASE_NUMERIC_COLS)} cột | Derived (tái tính): {len(DERIVED_COLS)} cột')

[1/7] Đọc dữ liệu gốc...
     Gốc: 4,178 dòng × 51 cột
     Base numeric: 30 cột | Derived (tái tính): 7 cột


In [3]:
# ============================================================
# 2. IMAN-CONOVER RANK CORRELATION MATCHING
# ============================================================
# Thuật toán:
#   1. Tính Spearman corr matrix của dữ liệu gốc → target
#   2. Cholesky decompose target → L
#   3. Sinh score matrix P từ rank của synthetic data
#   4. Tái sắp xếp rank của synthetic để khớp target
# Kết quả: marginal distribution GIỮ NGUYÊN, rank correlation → target

def nearest_psd(A, epsilon=1e-8):
    """Đảm bảo matrix là Positive Semi-Definite (cần thiết cho Cholesky)"""
    eigvals, eigvecs = np.linalg.eigh(A)
    eigvals = np.maximum(eigvals, epsilon)
    result = eigvecs @ np.diag(eigvals) @ eigvecs.T
    np.fill_diagonal(result, 1.0)
    return result


def iman_conover(synthetic_df, original_df, cols):
    """
    Iman-Conover (1982): Bảo toàn rank correlation structure của original.
    
    - Marginal distribution của từng cột KHÔNG thay đổi
    - Spearman correlation giữa các cột → khớp với original
    """
    print('     [IC] Tính Spearman corr matrix của dữ liệu gốc...')
    
    valid_cols = [c for c in cols if c in synthetic_df.columns and c in original_df.columns]
    n = len(synthetic_df)
    p = len(valid_cols)
    
    # Lấy Spearman corr của gốc
    corr_target = original_df[valid_cols].corr(method='spearman').values
    corr_target = nearest_psd(corr_target)  # Đảm bảo PSD
    
    # Cholesky
    print('     [IC] Cholesky decomposition...')
    L = np.linalg.cholesky(corr_target)
    
    # Tính rank matrix của synthetic (normalized)
    print('     [IC] Tính rank matrix của synthetic...')
    rank_matrix = np.zeros((n, p))
    for j, col in enumerate(valid_cols):
        rank_matrix[:, j] = rankdata(synthetic_df[col].values, method='ordinal')
    
    # Tạo score matrix: chuẩn hóa rank về normal scores
    # Van der Waerden scores: Φ⁻¹(rank / (n+1))
    from scipy.stats import norm
    score_matrix = norm.ppf(rank_matrix / (n + 1))
    
    # Tính corr của score matrix hiện tại
    S = np.cov(score_matrix.T)
    S_diag = np.sqrt(np.diag(S))
    S_diag[S_diag == 0] = 1.0
    corr_current = S / np.outer(S_diag, S_diag)
    corr_current = nearest_psd(corr_current)
    
    # Cholesky của corr hiện tại
    P = np.linalg.cholesky(corr_current)
    
    # Biến đổi score: T = score × P⁻¹ × L
    # → score mới có correlation structure = target
    print('     [IC] Áp dụng rank re-ordering...')
    P_inv = np.linalg.inv(P)
    T = score_matrix @ P_inv.T @ L.T
    
    # Tái sắp xếp từng cột của synthetic theo rank của T
    result_df = synthetic_df.copy()
    for j, col in enumerate(valid_cols):
        target_ranks = rankdata(T[:, j], method='ordinal').astype(int) - 1
        sorted_values = np.sort(synthetic_df[col].values)
        result_df[col] = sorted_values[target_ranks]
    
    return result_df


print('[IC Module] Iman-Conover function loaded.')

[IC Module] Iman-Conover function loaded.


In [4]:
# ============================================================
# 3. STRATIFIED SMOTE + NOISE (per Crop)
# ============================================================
def generate_smote_noise_stratified(df, n_samples, k=5, noise_level=0.015, log_cols=None):
    if log_cols is None:
        log_cols = []
    
    print(f'[2/7] Stratified SMOTE (k={k}) + Noise (scale={noise_level})...')
    
    crop_counts  = df['Crop Name'].value_counts()
    total_orig   = len(df)
    crop_target  = (crop_counts / total_orig * n_samples).round().astype(int)
    diff = n_samples - crop_target.sum()
    if diff != 0:
        crop_target[crop_counts.index[0]] += diff
    
    all_synthetic = []
    
    for crop_name, n_crop in crop_target.items():
        if n_crop <= 0:
            continue
        
        crop_df = df[df['Crop Name'] == crop_name].reset_index(drop=True)
        n_orig  = len(crop_df)
        
        if n_orig < 3:
            sampled = crop_df.sample(n=n_crop, replace=True).reset_index(drop=True)
            all_synthetic.append(sampled)
            continue
        
        data_num = crop_df[BASE_NUMERIC_COLS].fillna(crop_df[BASE_NUMERIC_COLS].median())
        
        # Log-transform cho cột skewness cao
        data_log = data_num.copy()
        for col in log_cols:
            if col in data_log.columns:
                data_log[col] = np.log1p(data_log[col].clip(lower=0))
        
        scaler      = MinMaxScaler()
        data_scaled = scaler.fit_transform(data_log)
        
        k_actual = min(k, n_orig - 1)
        nbrs = NearestNeighbors(n_neighbors=k_actual + 1).fit(data_scaled)
        
        parent_idxs = np.random.choice(n_orig, n_crop, replace=True)
        parents     = data_scaled[parent_idxs]
        
        indices       = nbrs.kneighbors(parents, return_distance=False)
        neighbor_idxs = np.array([np.random.choice(row[1:]) for row in indices])
        neighbors     = data_scaled[neighbor_idxs]
        
        ratios          = np.random.rand(n_crop, 1)
        synthetic_scaled = parents + ratios * (neighbors - parents)
        
        noise = np.random.normal(0, noise_level, synthetic_scaled.shape)
        synthetic_scaled = np.clip(synthetic_scaled + noise, 0, 1)
        
        synthetic_num = scaler.inverse_transform(synthetic_scaled)
        syn_df_num    = pd.DataFrame(synthetic_num, columns=BASE_NUMERIC_COLS)
        
        # Inverse log-transform
        for col in log_cols:
            if col in syn_df_num.columns:
                syn_df_num[col] = np.expm1(syn_df_num[col]).clip(lower=0)
        
        parent_cats = crop_df.iloc[parent_idxs][CATEGORICAL_COLS].reset_index(drop=True)
        all_synthetic.append(pd.concat([syn_df_num, parent_cats], axis=1))
    
    result = pd.concat(all_synthetic, ignore_index=True)
    print(f'     Đã sinh: {len(result):,} mẫu từ {len(crop_target)} loại cây')
    return result


synthetic_data = generate_smote_noise_stratified(
    data, n_samples=NUM_ROWS,
    k=K_NEIGHBORS, noise_level=NOISE_LEVEL, log_cols=LOG_COLS
)

[2/7] Stratified SMOTE (k=5) + Noise (scale=0.015)...
     Đã sinh: 20,000 mẫu từ 72 loại cây


In [5]:
# ============================================================
# 4. ÁP DỤNG IMAN-CONOVER → BẢO TOÀN CORRELATION STRUCTURE
# ============================================================
print('[3/7] Iman-Conover rank correlation matching...')

# Chỉ áp dụng trên BASE_NUMERIC_COLS (không gồm derived — sẽ tái tính sau)
# Và không áp dụng LOG_COLS vì chúng có phân phối đặc biệt
IC_COLS = [c for c in BASE_NUMERIC_COLS if c not in LOG_COLS]

synthetic_data = iman_conover(synthetic_data, data, IC_COLS)

print(f'     Iman-Conover hoàn tất trên {len(IC_COLS)} cột')

[3/7] Iman-Conover rank correlation matching...
     [IC] Tính Spearman corr matrix của dữ liệu gốc...
     [IC] Cholesky decomposition...
     [IC] Tính rank matrix của synthetic...
     [IC] Áp dụng rank re-ordering...
     Iman-Conover hoàn tất trên 28 cột


In [6]:
# ============================================================
# 5. HẬU XỬ LÝ LOGIC
# ============================================================
print('[4/7] Hậu xử lý Logic...')

# --- A. ĐỒNG BỘ CÂY TRỒNG & MÙA VỤ ---
cols_to_sync = ['Season', 'Transplant', 'Growth', 'Harvest']
synthetic_data = synthetic_data.drop(columns=cols_to_sync, errors='ignore')

sampled_rows = []
for crop in synthetic_data['Crop Name'].unique():
    idx          = synthetic_data[synthetic_data['Crop Name'] == crop].index
    orig_subset  = data[data['Crop Name'] == crop][cols_to_sync]
    if not orig_subset.empty:
        sampled = orig_subset.sample(n=len(idx), replace=True).set_index(idx)
        sampled_rows.append(sampled)

synthetic_data = pd.concat([synthetic_data, pd.concat(sampled_rows)], axis=1)

# --- B. RAINFALL CLIP (percentile P5/P95 per District×Season) ---
geo_climate = data.groupby(['District', 'Season'])['Rainfall'].agg(
    Rain_p05=lambda x: x.quantile(0.05),
    Rain_p95=lambda x: x.quantile(0.95)
).reset_index()
dist_climate = data.groupby('District')['Rainfall'].agg(
    Dist_p05=lambda x: x.quantile(0.05),
    Dist_p95=lambda x: x.quantile(0.95)
).reset_index()

synthetic_data = synthetic_data.merge(geo_climate, on=['District', 'Season'], how='left')
synthetic_data = synthetic_data.merge(dist_climate, on='District', how='left')
g_min = data['Rainfall'].quantile(0.01)
g_max = data['Rainfall'].quantile(0.99)
synthetic_data['Rain_p05'] = synthetic_data['Rain_p05'].fillna(synthetic_data['Dist_p05']).fillna(g_min)
synthetic_data['Rain_p95'] = synthetic_data['Rain_p95'].fillna(synthetic_data['Dist_p95']).fillna(g_max)
synthetic_data['Rainfall'] = synthetic_data['Rainfall'].clip(
    lower=synthetic_data['Rain_p05'], upper=synthetic_data['Rain_p95']
).round(2)
synthetic_data.drop(columns=['Rain_p05','Rain_p95','Dist_p05','Dist_p95'], inplace=True)

# --- C. CLIP ÂM & DOMAIN BOUNDS ---
non_neg = [
    'Rainfall','Soil_Moisture_mm','Nitrogen','Organic_Carbon',
    'Wind_Max','Wind_Mean','Heat_Stress_Days',
    'sm_surface','sm_rootzone','EVI','LAI','FPAR',
    'Avg_Salinity_Index','Bulk_Density'
]
for col in non_neg:
    if col in synthetic_data.columns:
        synthetic_data[col] = synthetic_data[col].clip(lower=0)

for col in [c for c in synthetic_data.columns if 'NDVI' in c and c in BASE_NUMERIC_COLS]:
    synthetic_data[col] = synthetic_data[col].clip(-1.0, 1.0)

if 'pH' in synthetic_data.columns:
    synthetic_data['pH'] = synthetic_data['pH'].clip(3.5, 9.0).round(2)
for col in ['Min Relative Humidity','Avg Humidity','Max Relative Humidity']:
    if col in synthetic_data.columns:
        synthetic_data[col] = synthetic_data[col].clip(0, 100).round(1)

def fix_min_mean_max(df, col_min, col_mean, col_max):
    if {col_min, col_mean, col_max}.issubset(df.columns):
        mask = df[col_min] > df[col_max]
        df.loc[mask, [col_min, col_max]] = df.loc[mask, [col_max, col_min]].values
        df[col_mean] = df[col_mean].clip(lower=df[col_min], upper=df[col_max])
    return df

synthetic_data = fix_min_mean_max(synthetic_data, 'Min Temp', 'Avg Temp', 'Max Temp')
synthetic_data = fix_min_mean_max(synthetic_data, 'Min Relative Humidity', 'Avg Humidity', 'Max Relative Humidity')
synthetic_data = fix_min_mean_max(synthetic_data, 'NDVI_Season_Min', 'NDVI_Season_Mean', 'NDVI_Season_Max')
synthetic_data[['Min Temp','Avg Temp','Max Temp']] = synthetic_data[['Min Temp','Avg Temp','Max Temp']].round(1)

if {'Wind_Mean','Wind_Max'}.issubset(synthetic_data.columns):
    mask_w = synthetic_data['Wind_Mean'] > synthetic_data['Wind_Max']
    synthetic_data.loc[mask_w,'Wind_Mean'] = synthetic_data.loc[mask_w,'Wind_Max'] * 0.8

# --- D. SOIL SUM = 100% ---
soil_cols = ['Sand','Silt','Clay']
if all(c in synthetic_data.columns for c in soil_cols):
    synthetic_data[soil_cols] = synthetic_data[soil_cols].clip(lower=0)
    total = synthetic_data[soil_cols].sum(axis=1).replace(0, 100)
    for c in soil_cols:
        synthetic_data[c] = (synthetic_data[c] / total * 100).round(2)
    synthetic_data['Clay'] = (100.0 - synthetic_data['Sand'] - synthetic_data['Silt']).round(2).clip(lower=0)

# --- E. FLAGS 2 CHIỀU ---
# FIX WIND: flag=1 trong gốc có Wind_Max=10.625 CỐ ĐỊNH
# → KHÔNG random(11,35) vì sẽ inflat correlation
WIND_FLAG_VALUE = data[data['is_extreme_Wind_Max'] == 1]['Wind_Max'].median()  # = 10.625
THRESHOLD_WIND  = data[data['is_extreme_Wind_Max'] == 0]['Wind_Max'].max()     # upper bound flag=0

if 'is_extreme_Wind_Max' in synthetic_data.columns:
    m1 = (synthetic_data['is_extreme_Wind_Max'] == 1) & (synthetic_data['Wind_Max'] < THRESHOLD_WIND)
    synthetic_data.loc[m1, 'Wind_Max'] = WIND_FLAG_VALUE   # gán đúng giá trị thực tế
    m0 = (synthetic_data['is_extreme_Wind_Max'] == 0) & (synthetic_data['Wind_Max'] >= WIND_FLAG_VALUE)
    synthetic_data.loc[m0, 'Wind_Max'] = np.random.uniform(2.0, THRESHOLD_WIND, size=m0.sum())
    synthetic_data['Wind_Max'] = synthetic_data['Wind_Max'].round(2)

THRESHOLD_HEAT = 43.0
if 'is_extreme_Heat_Stress_Days' in synthetic_data.columns:
    m1 = (synthetic_data['is_extreme_Heat_Stress_Days'] == 1) & (synthetic_data['Heat_Stress_Days'] < THRESHOLD_HEAT)
    synthetic_data.loc[m1, 'Heat_Stress_Days'] = np.random.uniform(THRESHOLD_HEAT, 65, size=m1.sum())
    m0 = (synthetic_data['is_extreme_Heat_Stress_Days'] == 0) & (synthetic_data['Heat_Stress_Days'] >= THRESHOLD_HEAT)
    synthetic_data.loc[m0, 'Heat_Stress_Days'] = np.random.uniform(0, 42.9, size=m0.sum())

if {'Extreme_Heat_Risk','Is_Extreme_Heat'}.issubset(synthetic_data.columns):
    mask_c = (synthetic_data['Is_Extreme_Heat'] == 1) & (synthetic_data['Extreme_Heat_Risk'] == 'Low Risk')
    synthetic_data.loc[mask_c, 'Extreme_Heat_Risk'] = 'High Risk'
    synthetic_data.loc[synthetic_data['Extreme_Heat_Risk'] == 'Low Risk', 'Is_Extreme_Heat'] = 0

[4/7] Hậu xử lý Logic...


In [7]:
# ============================================================
# 6. TÁI TÍNH DERIVED COLS VỚI CÔNG THỨC ĐÚNG
# ============================================================
print('[5/7] Tái tính Derived cols...')

# Yield = Production / Area
synthetic_data['Area']       = synthetic_data['Area'].clip(lower=1.0)
synthetic_data['Production'] = synthetic_data['Production'].clip(lower=0.0)
synthetic_data['Yield']      = (synthetic_data['Production'] / synthetic_data['Area']).round(4)

# CN_Ratio = OC / N
if 'CN_Ratio' in data.columns:
    synthetic_data['Nitrogen'] = synthetic_data['Nitrogen'].clip(lower=0.001)
    synthetic_data['CN_Ratio'] = (synthetic_data['Organic_Carbon'] / synthetic_data['Nitrogen']).round(4)

# Rootzone_Surface_Diff = sm_rootzone - sm_surface
if 'Rootzone_Surface_Diff' in data.columns:
    synthetic_data['Rootzone_Surface_Diff'] = (
        synthetic_data['sm_rootzone'] - synthetic_data['sm_surface']
    ).round(4)

# Moisture_Ratio = sm_rootzone / sm_surface
if 'Moisture_Ratio' in data.columns:
    sm_safe = synthetic_data['sm_surface'].clip(lower=0.001)
    synthetic_data['Moisture_Ratio'] = (synthetic_data['sm_rootzone'] / sm_safe).round(4)

# NDVI_Season_Range = Max - Min
if 'NDVI_Season_Range' in data.columns:
    synthetic_data['NDVI_Season_Range'] = (
        synthetic_data['NDVI_Season_Max'] - synthetic_data['NDVI_Season_Min']
    ).clip(lower=0).round(4)

# FIX: NDVI_Season_Std = Range × 0.454 (từ phân tích dữ liệu gốc: ratio mean=0.4544, std=0.029)
# Thêm nhiễu nhỏ để tránh bị deterministc hoàn toàn
if 'NDVI_Season_Std' in data.columns:
    NDVI_STD_RATIO_MEAN = data['NDVI_Season_Std'].div(
        data['NDVI_Season_Range'].replace(0, np.nan)
    ).mean()  # ~0.454
    NDVI_STD_RATIO_STD  = data['NDVI_Season_Std'].div(
        data['NDVI_Season_Range'].replace(0, np.nan)
    ).std()   # ~0.029
    noise_ratio = np.random.normal(NDVI_STD_RATIO_MEAN, NDVI_STD_RATIO_STD, len(synthetic_data))
    noise_ratio = np.clip(noise_ratio, 0.40, 0.55)  # giữ trong khoảng thực tế
    synthetic_data['NDVI_Season_Std'] = (
        synthetic_data['NDVI_Season_Range'] * noise_ratio
    ).round(4)
    print(f'     NDVI_Std ratio calibrated: mean={NDVI_STD_RATIO_MEAN:.4f}, std={NDVI_STD_RATIO_STD:.4f}')

# NDVI_Season_CV = Std / |Mean| (công thức đúng từ gốc)
if 'NDVI_Season_CV' in data.columns:
    ndvi_mean_safe = synthetic_data['NDVI_Season_Mean'].abs().replace(0, np.nan).fillna(0.001)
    synthetic_data['NDVI_Season_CV'] = (
        synthetic_data['NDVI_Season_Std'] / ndvi_mean_safe
    ).round(4)

# Rain_Temp_Ratio — tính theo tỷ lệ so với reference district+season
if 'Rain_Temp_Ratio' in data.columns:
    ref_ratio = data.groupby(['District','Season'])['Rain_Temp_Ratio'].median().reset_index()
    ref_ratio.columns = ['District','Season','RTR_ref']
    ref_rain  = data.groupby(['District','Season'])['Rainfall'].median().reset_index()
    ref_rain.columns  = ['District','Season','Rain_ref']
    synthetic_data = synthetic_data.merge(ref_ratio, on=['District','Season'], how='left')
    synthetic_data = synthetic_data.merge(ref_rain,  on=['District','Season'], how='left')
    safe_ref = synthetic_data['Rain_ref'].replace(0, np.nan).fillna(1)
    synthetic_data['Rain_Temp_Ratio'] = (
        synthetic_data['RTR_ref'] * synthetic_data['Rainfall'] / safe_ref
    ).round(4)
    synthetic_data.drop(columns=['RTR_ref','Rain_ref'], inplace=True)

[5/7] Tái tính Derived cols...
     NDVI_Std ratio calibrated: mean=0.4544, std=0.0292


In [8]:
# ============================================================
# 7. ĐỒNG BỘ CATEGORICAL LABELS
# ============================================================
print('[6/7] Đồng bộ Categorical labels...')

if 'Dominant_Soil_Texture' in synthetic_data.columns:
    conds = [(synthetic_data['Clay'] >= 40),(synthetic_data['Sand'] >= 50),(synthetic_data['Silt'] >= 50)]
    synthetic_data['Dominant_Soil_Texture'] = np.select(conds, ['Clayey','Sandy','Silty'], default='Loamy')

if 'pH_Suitability' in synthetic_data.columns:
    conds_ph = [(synthetic_data['pH'] < 5.5),(synthetic_data['pH'] > 7.5)]
    synthetic_data['pH_Suitability'] = np.select(conds_ph, ['Acidic','Alkaline'], default='Optimal')

if 'Water_Availability_Cat' in synthetic_data.columns:
    p33 = data['sm_rootzone'].quantile(0.33)
    p66 = data['sm_rootzone'].quantile(0.66)
    orig_labels = set(data['Water_Availability_Cat'].unique())
    # Detect label set (Optimal/High/Low vs High/Medium/Low)
    if 'Optimal' in orig_labels:
        choices_w = ['High','Optimal','Low']
    else:
        choices_w = ['High','Medium','Low']
    conds_w = [(synthetic_data['sm_rootzone'] >= p66),(synthetic_data['sm_rootzone'] >= p33)]
    synthetic_data['Water_Availability_Cat'] = np.select(conds_w, choices_w[:2], default=choices_w[2])

[6/7] Đồng bộ Categorical labels...


In [9]:
# ============================================================
# 8. LƯU FILE & BÁO CÁO ĐÁNH GIÁ
# ============================================================
print('[7/7] Lưu file & đánh giá...')

original_cols = [c for c in data.columns if c in synthetic_data.columns]
extra_cols    = [c for c in synthetic_data.columns if c not in original_cols]
synthetic_data = synthetic_data[original_cols + extra_cols]

output_file = 'Agri_Data_SMOTE_Noise_v3.csv'
synthetic_data.to_csv(output_file, index=False)

# ---- BÁO CÁO SO SÁNH ----
num_cols = data.select_dtypes(include=np.number).columns.tolist()
num_cols_common = [c for c in num_cols if c in synthetic_data.columns]

corr_orig = data[num_cols_common].corr(method='spearman')
corr_new  = synthetic_data[num_cols_common].corr(method='spearman')

# Frobenius norm
diff_mat  = corr_orig.values - corr_new.values
frob_new  = np.sqrt((diff_mat**2).sum())

# Per-column MAE
col_mae = {c: (corr_orig[c] - corr_new[c]).abs().mean() for c in num_cols_common}
top_drift = sorted(col_mae.items(), key=lambda x: -x[1])[:5]

print(f"""
{'='*55}
[THÀNH CÔNG] Đã lưu: {output_file}
{'='*55}
Tổng số dòng     : {len(synthetic_data):,}
Yield mean       : {synthetic_data['Yield'].mean():.4f}  (Gốc: {data['Yield'].mean():.4f})
Yield std        : {synthetic_data['Yield'].std():.4f}  (Gốc: {data['Yield'].std():.4f})
Area mean        : {synthetic_data['Area'].mean():.1f}  (Gốc: {data['Area'].mean():.1f})
Production mean  : {synthetic_data['Production'].mean():.1f}  (Gốc: {data['Production'].mean():.1f})

--- Correlation Structure (Spearman) ---
Frobenius norm   : {frob_new:.4f}  (v2 improved: ~2.51)
""")

print('Top 5 cột còn drift nhiều nhất:')
for col, mae in top_drift:
    print(f'  {col:<35}: MAE = {mae:.4f}')

print('\n--- Kiểm tra tính nhất quán Derived cols ---')
y_check = ((synthetic_data['Production']/synthetic_data['Area']).round(4) == synthetic_data['Yield']).mean()
print(f'  Yield = Prod/Area       : {y_check*100:.1f}%')
if 'Rootzone_Surface_Diff' in synthetic_data.columns:
    d_check = ((synthetic_data['sm_rootzone']-synthetic_data['sm_surface']).round(4) == synthetic_data['Rootzone_Surface_Diff']).mean()
    print(f'  Rootzone_Diff consistent: {d_check*100:.1f}%')
if 'CN_Ratio' in synthetic_data.columns:
    cn_check = ((synthetic_data['Organic_Carbon']/synthetic_data['Nitrogen']).round(4) == synthetic_data['CN_Ratio']).mean()
    print(f'  CN_Ratio consistent     : {cn_check*100:.1f}%')

print('\n--- Wind Flag check ---')
wc = synthetic_data['Wind_Max'].corr(synthetic_data['is_extreme_Wind_Max'])
print(f'  Wind_Max vs flag corr: {wc:.3f}  (Gốc: 0.390, v2: 0.823 [sai])')
print(f'  Wind_Max max (new)   : {synthetic_data["Wind_Max"].max():.3f}  (Gốc: 10.625)')

print('\n--- NDVI_CV check ---')
ndvi_mae = (corr_orig['NDVI_Season_CV'] - corr_new['NDVI_Season_CV']).abs().mean()
print(f'  NDVI_CV corr MAE: {ndvi_mae:.4f}  (v2: 0.1357)')

[7/7] Lưu file & đánh giá...

[THÀNH CÔNG] Đã lưu: Agri_Data_SMOTE_Noise_v3.csv
Tổng số dòng     : 20,000
Yield mean       : 4.0025  (Gốc: 4.1786)
Yield std        : 4.8861  (Gốc: 5.7743)
Area mean        : 8410.5  (Gốc: 8917.0)
Production mean  : 14789.0  (Gốc: 15841.2)

--- Correlation Structure (Spearman) ---
Frobenius norm   : 1.1116  (v2 improved: ~2.51)

Top 5 cột còn drift nhiều nhất:
  Moisture_Ratio                     : MAE = 0.0397
  NDVI_Season_Min                    : MAE = 0.0327
  EVI                                : MAE = 0.0320
  Heat_Stress_Days                   : MAE = 0.0288
  NDVI_Season_Range                  : MAE = 0.0256

--- Kiểm tra tính nhất quán Derived cols ---
  Yield = Prod/Area       : 100.0%
  Rootzone_Diff consistent: 100.0%
  CN_Ratio consistent     : 100.0%

--- Wind Flag check ---
  Wind_Max vs flag corr: 0.427  (Gốc: 0.390, v2: 0.823 [sai])
  Wind_Max max (new)   : 10.630  (Gốc: 10.625)

--- NDVI_CV check ---
  NDVI_CV corr MAE: 0.0227  (v2: 0.13